# 04 — Continue the LoRA adapter with GRPO

GRPO samples a group of candidate explanations for each self-contained task,
scores them, and reinforces answers that are better **relative to the group**.

```
                      ┌─ candidate A ─ rewards ─┐
mechanism task ──────┼─ candidate B ─ rewards ─┼─ relative advantages ─ LoRA update
                      ├─ candidate C ─ rewards ─┤
                      └─ candidate D ─ rewards ─┘
```

This notebook continues the SFT LoRA adapter. It does not restart from the base
model, and it never trains against numerical target values.

## Important execution note

The semantic reward calls `gpt-5.6-terra`. Use a single training process in this
teaching notebook; the batched judge cache is intentionally simple and auditable.
Each unique completion is paid API work, so begin with a short run and inspect the
logged rewards before increasing `MAX_STEPS` in the configuration cell.

## Configuration

All models, rewards, training hyperparameters, paths, and Hub repositories are
explicit here. `OPENAI_API_KEY` remains an environment secret. Hugging Face uses
`hf auth login`; the optional token line may be uncommented but never populated
with a literal token inside the notebook.

In [ ]:
import os

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from accelerate import PartialState
from datasets import load_from_disk
from IPython.display import JSON, Markdown, display
from peft import PeftModel
from trl import GRPOConfig, GRPOTrainer

from science_course.devices import clear_device_cache, detect_runtime
from science_course.hub import require_hf_namespace
from science_course.judge import (
    configure_mechanism_judge,
    mechanism_judge_reward,
)
from science_course.modeling import (
    generate_text,
    load_causal_lm,
    load_tokenizer,
    render_prompt,
)
from science_course.rewards import (
    concept_coverage_reward,
    evidence_grounding_reward,
    format_reward,
)
from science_course.teacher import DEFAULT_TEACHER_MODEL, require_openai_key
from science_course.versions import require_training_stack

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Runtime and artifacts
ENABLE_MPS_FALLBACK = True
if ENABLE_MPS_FALLBACK:
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
MODEL_ID = "google/gemma-4-E4B-it"
SFT_ADAPTER_SOURCE = ROOT / "artifacts" / "gemma4-mechanism-sft"
GRPO_ADAPTER = ROOT / "artifacts" / "gemma4-mechanism-grpo"
GRPO_DATA = ROOT / "data" / "processed" / "grpo"
RESUME_FROM_CHECKPOINT = None

# Semantic judge
JUDGE_MODEL = DEFAULT_TEACHER_MODEL
JUDGE_CACHE = ROOT / "results" / "grpo_judge_cache.jsonl"

# GRPO
MAX_STEPS = 25
LEARNING_RATE = 5e-6
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
NUM_GENERATIONS = 4
NUM_GENERATIONS_EVAL = 4
MAX_COMPLETION_LENGTH = 256
TEMPERATURE = 0.8
TOP_P = 0.95
BETA = 0.0
REWARD_WEIGHTS = [0.10, 0.15, 0.15, 0.60]
WARMUP_RATIO = 0.05
EVAL_STEPS = 10
SAVE_STEPS = 10
LOGGING_STEPS = 1
NUM_COMPLETIONS_TO_PRINT = 4
RANDOM_SEED = 17

# Hugging Face publication: every GRPO checkpoint plus the final adapter
DATASET_HF_REPO = "lamm-mit/scientific-sft-grpo-data"
SFT_HF_REPO = "lamm-mit/scientific-sft-grpo-sft"
GRPO_HF_REPO = "lamm-mit/scientific-sft-grpo-grpo"
PUSH_TO_HUB = True
HUB_PRIVATE_REPO = False
HF_TOKEN = None
# HF_TOKEN = os.environ["HF_TOKEN"]  # Optional; prefer `hf auth login`.

require_openai_key()
versions = require_training_stack()
runtime = detect_runtime()
if JUDGE_MODEL != "gpt-5.6-terra":
    raise RuntimeError("The tested semantic judge is gpt-5.6-terra.")
if PartialState().num_processes != 1:
    raise RuntimeError("This API-judged teaching run requires WORLD_SIZE=1.")
configure_mechanism_judge(
    model=JUDGE_MODEL,
    cache_path=JUDGE_CACHE,
)
if PUSH_TO_HUB:
    require_hf_namespace(GRPO_HF_REPO, token=HF_TOKEN)

display(
    JSON(
        {
            "runtime": runtime.as_dict(),
            "versions": versions,
            "model": MODEL_ID,
            "sft_adapter": str(SFT_ADAPTER_SOURCE),
            "grpo_hub_repo": GRPO_HF_REPO,
        }
    )
)

## 1. Load the GRPO data and the trainable SFT adapter

Hidden columns remain in the dataset because custom reward functions receive them.
The model receives only `prompt`. Loading the SFT adapter with `is_trainable=True`
ensures GRPO updates the same LoRA parameters.

In [ ]:
if not GRPO_DATA.exists():
    raise RuntimeError("Run notebook 02 to create the GRPO dataset.")
if isinstance(SFT_ADAPTER_SOURCE, Path) and not SFT_ADAPTER_SOURCE.exists():
    raise RuntimeError("Run notebook 03 to create the SFT adapter.")

grpo = load_from_disk(GRPO_DATA)
if len(grpo["train"]) == 0:
    raise RuntimeError("The GRPO train split is empty.")

tokenizer = load_tokenizer(MODEL_ID, token=HF_TOKEN)
clear_device_cache(runtime)
base_model = load_causal_lm(MODEL_ID, runtime, token=HF_TOKEN)
model = PeftModel.from_pretrained(
    base_model,
    SFT_ADAPTER_SOURCE,
    is_trainable=True,
    token=HF_TOKEN,
)
model.print_trainable_parameters()
print(grpo)

## 2. Establish a pre-GRPO behavior snapshot

We keep the same held-out prompt before and after training. This is a qualitative
teaching comparison, not a claim of scientific benchmark performance.

In [ ]:
held_out_pool = grpo["test"] if len(grpo["test"]) else grpo["validation"]
if not len(held_out_pool):
    raise RuntimeError("A validation or test task is required for comparison.")
held_out = held_out_pool[0]
held_out_prompt = render_prompt(tokenizer, held_out["prompt"])
before_grpo = generate_text(model, tokenizer, held_out_prompt, runtime)
display(Markdown("### SFT policy\n```text\n" + before_grpo + "\n```"))

## 3. Configure mechanism-sensitive GRPO

Reward weights intentionally make causal judgment dominant:

- format: 0.10
- task-grounded evidence: 0.15
- rubric concept coverage: 0.15
- batched `gpt-5.6-terra` mechanism judge: 0.60

Deterministic checks constrain obvious failure modes; the semantic judge evaluates
causal direction, intermediate steps, support, and scientific coherence.

In [ ]:
grpo_args = GRPOConfig(
    output_dir=str(GRPO_ADAPTER),
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_generations=NUM_GENERATIONS,
    num_generations_eval=NUM_GENERATIONS_EVAL,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    beta=BETA,
    reward_weights=REWARD_WEIGHTS,
    scale_rewards="group",
    loss_type="dapo",
    remove_unused_columns=False,
    chat_template_kwargs={"enable_thinking": False},
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=None,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    log_completions=True,
    num_completions_to_print=NUM_COMPLETIONS_TO_PRINT,
    report_to="none",
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=GRPO_HF_REPO,
    hub_strategy="all_checkpoints",
    hub_private_repo=HUB_PRIVATE_REPO,
    hub_token=HF_TOKEN,
    hub_always_push=True,
    bf16=runtime.trainer_bf16,
    fp16=runtime.trainer_fp16,
    use_cpu=runtime.use_cpu,
    use_vllm=False,
    seed=RANDOM_SEED,
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_args,
    reward_funcs=[
        format_reward,
        evidence_grounding_reward,
        concept_coverage_reward,
        mechanism_judge_reward,
    ],
    train_dataset=grpo["train"],
    eval_dataset=grpo["validation"],
    processing_class=tokenizer,
)

## 4. Train and inspect reward dynamics

A rising total reward is not enough: inspect the individual components. If format
rises while semantic reward stagnates, the policy is learning the shell but not a
better mechanism.

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)
trainer.save_model(GRPO_ADAPTER)
tokenizer.save_pretrained(GRPO_ADAPTER)
if PUSH_TO_HUB:
    hub_result = trainer.push_to_hub(
        commit_message="Complete mechanism GRPO training"
    )
    print(f"Published final adapter and all checkpoints: {hub_result}")
display(JSON(train_result.metrics))

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
reward_columns = [
    column
    for column in history.columns
    if column == "reward"
    or (column.startswith("rewards/") and column.endswith("/mean"))
]
if reward_columns:
    history[["step", *reward_columns]].dropna(how="all", subset=reward_columns).plot(
        x="step", y=reward_columns, figsize=(12, 5), marker="o"
    )
    plt.title("GRPO reward components")
    plt.ylabel("reward")
    plt.tight_layout()
    plt.show()
else:
    print("Reward columns available after this run:", sorted(history.columns))

## 5. Compare the same unseen mechanism task

Look for an ordered, supported causal explanation—not a particular phrase. The
source document is unnecessary because the task itself contains the relevant
observations.

In [ ]:
after_grpo = generate_text(trainer.model, tokenizer, held_out_prompt, runtime)
display(Markdown("### Before GRPO\n```text\n" + before_grpo + "\n```"))
display(Markdown("### After GRPO\n```text\n" + after_grpo + "\n```"))
display(Markdown("### Hidden reference (for instructor inspection)"))
display(
    JSON(
        {
            "mechanism_steps": held_out["mechanism_steps"],
            "causal_links": held_out["causal_links"],
            "reference_answer": held_out["reference_answer"],
        }
    )
)

## Result and responsible interpretation

`artifacts/gemma4-mechanism-grpo/` is the final LoRA adapter. The final adapter and
every saved checkpoint are also published to
`lamm-mit/scientific-sft-grpo-grpo`. At inference, attach it to the same base model
and submit any self-contained scientific mechanism task.

For real research, add expert adjudication, inter-rater reliability, domain-specific
test sets, ablations for each reward component, contamination checks, and multiple
seeds. An LLM judge is scalable supervision—not ground truth.